# Consistency Check for Imports

In [1]:
# Build a dictionary to map (file, row_index) → sentence metadata
def build_sentence_metadata_lookup(corpus_sentences):
    """
    corpus_sentences: list of Sentence objects
    Returns: {(file_name, row_index): sentence_metadata_dict}
    """
    lookup = {}
    for sent in corpus_sentences:
        for token in sent.get_tokens():
            if token.id is not None:
                # Use token.id as row_index
                lookup[(sent.file_name, int(token.id))] = sent.metadata
    return lookup


In [2]:
def assign_newpart(df):
    """
    Forward-fill the 'newpart' column across all rows.
    Only propagate values that are not empty or '_'.
    Returns a list of newpart values for each row.
    """
    newparts = []
    last_newpart = "_"

    for value in df.get('newpart', pd.Series("_")).astype(str):
        value = value.strip()
        if value and value != "_":  # Only consider "real" markers
            last_newpart = value
        newparts.append(last_newpart)

    return newparts


In [3]:
from tabulate import tabulate
import os

def _write_report(issues_list, folder, prefix, show_context=True):
    """
    Writes issues in a clear, philologist-friendly report.
    Groups issues by 'newpart' and formats them in tables.
    
    Args:
        issues_list: list of issue dicts
        folder: folder path to write the report
        prefix: filename prefix
        show_context: if True, include a few tokens of context
    """
    if not folder:
        return
    os.makedirs(folder, exist_ok=True)

    if not issues_list:
        filename = f"{prefix}_empty.txt"
    else:
        filename = f"{prefix}_{issues_list[0]['file']}.txt"
    
    path = os.path.join(folder, filename)

    # Group issues by newpart
    grouped = {}
    for issue in issues_list:
        np = issue.get('newpart', '_')
        if np not in grouped:
            grouped[np] = []
        grouped[np].append(issue)

    with open(path, 'w', encoding='utf-8') as f:
        if not issues_list:
            f.write("✅ No issues detected.\n")
            return

        f.write(f"Report: {prefix}\n")
        f.write(f"Total issues: {len(issues_list)}\n")
        f.write("="*80 + "\n\n")

        for newpart, issues in grouped.items():
            f.write(f"NEWPART: {newpart}\n")
            f.write(f"File: {issues[0]['file']}\n")
            f.write("-"*80 + "\n")

            # Table header
            headers = ["Row", "Lemma", "POS", "Meaning", "Issue Type", "Description"]
            table = []

            for issue in issues:
                context = ""
                if show_context and 'transcription' in issue:
                    context = issue.get('transcription', '')[:50] + ("..." if len(issue.get('transcription', '')) > 50 else "")
                table.append([
                    issue.get('row_index', ''),
                    issue.get('lemma', ''),
                    issue.get('pos', ''),
                    issue.get('meaning', ''),
                    issue.get('issue_type', ''),
                    issue.get('issue_desc', '') + (f" | Context: {context}" if context else "")
                ])
            
            f.write(tabulate(table, headers=headers, tablefmt="grid"))
            f.write("\n\n")


In [4]:
def _write_new_sense_report_with_metadata(s, folder, prefix, sentence_lookup):
    if not folder:
        return
    os.makedirs(folder, exist_ok=True)
    filename = f"{prefix}.txt" if s else f"{prefix}_empty.txt"
    path = os.path.join(folder, filename)
    with open(path, 'w', encoding='utf-8') as f:
        if not s:
            f.write("✅ No new senses detected.\n")
        else:
            for item in s:
                # Fetch metadata if available
                metadata = sentence_lookup.get((item['file'], item['row_index']), {})
                newpart_info = metadata.get('newpart', '_')
                f.write(
                    f"File: {item['file']} | Row {item['row_index']} | NEWPART: {newpart_info} | "
                    f"Lemma: {item['lemma']} | UPOS: {item['upos']} | New Sense: {item['new_sense']} | "
                    f"Other Senses: {item['other_senses']}\n"
                )


In [5]:
def _write_general_unsorted_lemma_report(s, folder, prefix, sentence_lookup):
    if not folder:
        return
    os.makedirs(folder, exist_ok=True)
    filename = f"{prefix}.txt" if s else f"{prefix}_empty.txt"
    path = os.path.join(folder, filename)
    with open(path, 'w', encoding='utf-8') as f:
        if not s:
            f.write("✅ No general ignoring edited lemma issues detected.\n")
        else:
            for item in s:
                f.write(
                    f"File: {item['file']} | Row {item['row_index']} | Lemma: {item['lemma']} | "
                    f"UPOS: {item['upos']} | New Sense: {item['new_sense']} | "
                    f"Other Senses: {item['other_senses']} | NEWPART: {item.get('newpart', '_')}\n"
                )


#above is fixed in the next cell

In [6]:
# mptf_validator_clean.py - Jupyter-friendly, no debugging

import os
import re
import pandas as pd
from collections import defaultdict
import unicodedata



CSV_EXTS = ('.csv', '.tsv', '.xlsx', '.xls')

# --- Helper functions -----------------------------------------------------

def _is_placeholder_meaning(text):
    """
    Returns True if the meaning is a placeholder that should be ignored
    during lemma-sense validation.
    Examples of placeholders: 'n/a', '§', '-', '$', ','.
    """
    if not text:
        return True
    text = str(text).strip().lower()
    return text in {'n/a', '§', '-', '$', ','}


def _read_table(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext in ('.xlsx', '.xls'):
            return pd.read_excel(path, dtype=str, keep_default_na=False, engine='openpyxl')
        else:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                sample = f.read(4096)
                sep = '\t' if sample.count('\t') >= sample.count(',') else ','
            return pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False, engine='python', on_bad_lines='skip')
    except Exception:
        return None

def _norm_str(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

# --- Checks ---------------------------------------------------------------
def _normalize_translit_backslash(translit):
    """
    Normalize backslashes in transliteration:
    - Remove them if they are the first or last character.
    - Replace them with underscores '_' if they occur elsewhere.
    """
    if not isinstance(translit, str) or translit.strip() == '':
        return translit

    # Remove leading and trailing backslashes
    translit = translit.strip('\\')
    translit = translit.strip('/')

    # Replace any remaining (internal) backslashes with underscores
    translit = re.sub(r'\\', '_', translit)
    translit = re.sub(r'/', '_', translit)

    return translit



def load_lemma_dictionary(dict_path):
    """
    Load lemma → allowed senses from a CSV/Excel dictionary.

    Handles:
    - Multi-sense entries separated by '|' or ','
    - Numeric suffixes in lemmata
    """
    import pandas as pd
    df = pd.read_excel(dict_path, dtype=str, keep_default_na=False, engine='openpyxl') \
        if dict_path.lower().endswith(('.xls', '.xlsx')) else pd.read_csv(dict_path, dtype=str, keep_default_na=False)
    
    lemma_col = next((c for c in df.columns if 'lemma' in c.lower()), None)
    sense_col = next((c for c in df.columns if 'sense' in c.lower() or 'meaning' in c.lower()), None)
    if not lemma_col or not sense_col:
        raise ValueError("Dictionary must have lemma and sense columns")
    
    lemma_map = {}
    for _, row in df.iterrows():
        lemma = _normalize_sense_text(row.get(lemma_col, ''))
        senses_raw = row.get(sense_col, '')
        if not senses_raw or str(senses_raw).strip() == '':
            senses = set()
        else:
            parts = re.split(r'\s*\|\s*|\s*,\s*', str(senses_raw))
            senses = {_normalize_sense_text(p) for p in parts if p.strip()}
        if lemma:
            lemma_map[lemma] = senses
    return lemma_map

def _strip_trailing_digits(s):
    """Remove numeric suffixes from lemma for flexible matching."""
    return re.sub(r'\d+$', '', s)


def _is_exception_token(value):
    """
    Returns True if the value is an exceptional
    that should be exempt from validation.
    """
    return str(value).strip() in {",", "$", "§"}


def _has_disallowed_punct(value):
    """
    Returns True if the value contains disallowed punctuation.
    Letters, digits, underscores, hyphens, and transcription marks are allowed.
    """
    if not value:
        return False

    allowed_marks = "=_-[]()" + "ˈʼʾʿˊˋˌː"

    for ch in str(value):
        if ch.isalpha() or ch.isdigit() or ch in allowed_marks:
            continue
        return True  # any other character is considered disallowed punctuation

    return False

def _contains_parentheses(translit):
    return bool(re.search(r'[\(\)\[\]\{\}]', translit))

def _translit_only_suffix_like(translit):
    t = translit.strip()
    if not t:
        return False
    # Skip if all chars are uppercase or known exception
    if all(not c.islower() for c in t):
        return False
    vowel_chars = re.compile(r'[aeiouAEIOUāēīōūyʾʿ]')
    if len(t) <= 4 and not vowel_chars.search(t):
        return True
    if re.fullmatch(r"[A-Za-zʾʿˈˋˌː\-\|/]+", t) and len(t) <= 7 and not vowel_chars.search(t):
        return True
    return False

def _is_prefixed_problem(transcription, translit, prefixes=None):
    if prefixes is None:
        prefixes = ['a', 'hu', 'an', 'ab']
    if not transcription or not translit:
        return False
    # Skip long transliterations or ones with non-lowercase chars
    if len(translit) > 8 or any(not c.islower() for c in translit):
        return False
    for prefix in prefixes:
        if transcription.startswith(prefix):
            m = re.match(r"(.{1,4}).*?\1", translit)
            if m:
                return True
    return False

def _ends_with_comparative(transcription):
    allowed_tokens = ('andar', 'aleksandar', 'brādar', 'hamdar', 'indar', 'mādar', 'pidar', 'nōdar', 'ušēdar', 'tištar', 'tuštar', 
                      'xrafstar', 'ǰuttar', 'axtar', 'abāxtar', 'tar', 'uštar', 'duxtar', 'dar', 'astar', 'ranguštar', 'wattar', 'ratuštar', 'ēdar', 'wāstar')
    if transcription in allowed_tokens:
        return False
    return bool(re.search(r'(dar|tar)$', transcription))

def _is_verb_pos(pos):
    return (pos or '').upper() in ('VERB', 'V', 'VB', 'AUX')

def _lemma_missing_infinitive(lemma, transcription, pos):
    if not _is_verb_pos(pos):
        return False
    l = (lemma or '').strip()
    if not l:
        return True
    if l.endswith(('an', 'ān', 'īdan', 'idan', 'dan')):
        return False
    if re.search(r'(īd|īdah|īdan|ān|tn)$', transcription):
        return True
    return False

def _enclitic_hyphen_issue(transcription):
    if not transcription:
        return False
    return bool(re.search(r'[A-Za-z͡ˈʼʾʿ0-9]+-[A-Za-z͡ˈʼʾʿ0-9]+', transcription)) and ('=' not in transcription)

def _wordend_marker_inconsistent(translit):
    return bool(translit and ("'" in translit and "ˈ" in translit))

def _has_restored_or_question(translit):
    return bool(translit and re.search(r'[\?\*]', translit))


import unicodedata

def _normalize_sense_text(text):
    """
    Normalize a lemma or sense string for consistent comparison.

    This function:
      - Converts the input to string (if not already)
      - Applies Unicode normalization (NFC)
      - Strips leading/trailing whitespace
      - Converts to lowercase

    Used for:
      - Normalizing lemma names when reading from dictionary files
      - Normalizing sense entries (e.g., meanings like "This" → "this")

    Parameters
    ----------
    text : str
        Input text (lemma or sense)

    Returns
    -------
    str
        A clean, normalized version of the input suitable for
        dictionary comparison and equality checks.
    """
    if text is None:
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()




def load_lemma_dictionary(dict_path):
    """
    Load a lemma-to-senses mapping from an Excel file.

    The Excel file must contain columns:
      - 'lemma'
      - 'related_senses' (with meanings separated by '|')

    Returns
    -------
    dict
        A dictionary where keys are normalized lemma strings and
        values are sets of normalized allowed senses.
    """
    df_dict = pd.read_excel(dict_path, dtype=str, keep_default_na=False, engine='openpyxl')
    lemma_sense_map = {}

    for _, row in df_dict.iterrows():
        lemma = _normalize_sense_text(row.get('lemma', ''))
        senses_raw = row.get('related_senses', '')
        senses = {_normalize_sense_text(s) for s in re.split(r'\s*\|\s*', senses_raw) if s.strip()}
        lemma_sense_map[lemma] = senses

    return lemma_sense_map

def _normalize_verb_sense_text(meaning, pos):
    """
    Normalize a sense string for comparison, especially for verbs.

    This function:
      - Converts the input to lowercase and strips whitespace
      - Removes leading 'to ' if the part of speech is a verb
      - Splits multi-sense entries separated by '|' or ',' into a list of normalized senses

    Parameters
    ----------
    meaning : str
        The sense/meaning string from the corpus or dictionary
    pos : str
        Part-of-speech tag of the token (e.g., 'VERB', 'NOUN')

    Returns
    -------
    set[str]
        A set of normalized senses, suitable for comparison against a dictionary
        Example:
            input: "to call | to summon", pos='VERB'
            output: {"call", "summon"}
    """
    if not meaning:
        return set()

    # Normalize text
    norm = meaning.strip().lower()

    # Remove leading 'to ' if POS is verb
    if _is_verb_pos(pos) and norm.startswith('to '):
        norm = norm[3:].strip()

    # Split multi-sense entries on | or ,
    parts = re.split(r'\s*\|\s*|\s*,\s*', norm)
    # Remove empty entries and normalize
    senses = {_normalize_sense_text(p) for p in parts if p.strip()}

    return senses


import os
import re
import pandas as pd
from collections import defaultdict

CSV_EXTS = ('.csv', '.tsv', '.xlsx', '.xls')



In [1]:
import os
import re
import pandas as pd
import unicodedata
new_lemma_added = set()
new_sense_added = []

CSV_EXTS = ('.csv', '.tsv', '.xlsx', '.xls')

def _lemma_missing_infinitive(lemma, transcription, pos):
    if not _is_verb_pos(pos):
        return False
    l = (lemma or '').strip()
    if not l:
        return True
    if l.endswith(('an', 'ān', 'īdan', 'idan', 'dan')):
        return False
    if re.search(r'(īd|īdah|īdan|ān|tn)$', transcription):
        return True
    return False


def _is_placeholder_meaning(text):
    """
    Returns True if the meaning is a placeholder that should be ignored
    during lemma-sense validation.
    Examples of placeholders: 'n/a', '§', '-', '$', ','.
    """
    if not text:
        return True
    text = str(text).strip().lower()
    return text in {'n/a', '§', '-', '$', ','}


def _strip_trailing_digits(s):
    """Remove numeric suffixes from lemma for flexible matching."""
    return re.sub(r'\d+$', '', s)


def _normalize_verb_sense_text(meaning, pos):
    """
    Normalize a sense string for comparison, especially for verbs.

    This function:
      - Converts the input to lowercase and strips whitespace
      - Removes leading 'to ' if the part of speech is a verb
      - Splits multi-sense entries separated by '|' or ',' into a list of normalized senses

    Parameters
    ----------
    meaning : str
        The sense/meaning string from the corpus or dictionary
    pos : str
        Part-of-speech tag of the token (e.g., 'VERB', 'NOUN')

    Returns
    -------
    set[str]
        A set of normalized senses, suitable for comparison against a dictionary
        Example:
            input: "to call | to summon", pos='VERB'
            output: {"call", "summon"}
    """
    if not meaning:
        return set()

    # Normalize text
    norm = meaning.strip().lower()

    # Remove leading 'to ' if POS is verb
    if _is_verb_pos(pos) and norm.startswith('to '):
        norm = norm[3:].strip()

    # Split multi-sense entries on | or ,
    parts = re.split(r'\s*\|\s*|\s*,\s*', norm)
    # Remove empty entries and normalize
    senses = {_normalize_sense_text(p) for p in parts if p.strip()}

    return senses


def _enclitic_hyphen_issue(transcription):
    if not transcription:
        return False
    return bool(re.search(r'[A-Za-z͡ˈʼʾʿ0-9]+-[A-Za-z͡ˈʼʾʿ0-9]+', transcription)) and ('=' not in transcription)

def _wordend_marker_inconsistent(translit):
    return bool(translit and ("'" in translit and "ˈ" in translit))

def _has_restored_or_question(translit):
    return bool(translit and re.search(r'[\?\*]', translit))


def _is_verb_pos(pos):
    return (pos or '').upper() in ('VERB', 'V', 'VB', 'AUX')


def _ends_with_comparative(transcription):
    allowed_tokens = ('andar', 'aleksandar', 'brādar', 'hamdar', 'indar', 'mādar', 'pidar', 'nōdar', 'ušēdar', 'tištar', 'tuštar', 
                      'xrafstar', 'ǰuttar', 'axtar', 'abāxtar', 'tar', 'uštar', 'duxtar', 'dar', 'astar', 'ranguštar', 'wattar', 'ratuštar', 'ēdar', 'wāstar')
    if transcription in allowed_tokens:
        return False
    return bool(re.search(r'(dar|tar)$', transcription))


def _has_disallowed_punct(value):
    """
    Returns True if the value contains disallowed punctuation.
    Letters, digits, underscores, hyphens, and transcription marks are allowed.
    """
    if not value:
        return False

    allowed_marks = "=_-[]()" + "ˈʼʾʿˊˋˌː"

    for ch in str(value):
        if ch.isalpha() or ch.isdigit() or ch in allowed_marks:
            continue
        return True  # any other character is considered disallowed punctuation

    return False

def _is_exception_token(value):
    """
    Returns True if the value is an exceptional
    that should be exempt from validation.
    """
    return str(value).strip() in {",", "$", "§"}



def _normalize_translit_backslash(translit):
    """
    Normalize backslashes in transliteration:
    - Remove them if they are the first or last character.
    - Replace them with underscores '_' if they occur elsewhere.
    """
    if not isinstance(translit, str) or translit.strip() == '':
        return translit

    # Remove leading and trailing backslashes
    translit = translit.strip('\\')
    translit = translit.strip('/')

    # Replace any remaining (internal) backslashes with underscores
    translit = re.sub(r'\\', '_', translit)
    translit = re.sub(r'/', '_', translit)

    return translit



def assign_newpart(df):
    """
    Forward-fill the 'newpart' column across all rows.
    Only propagate values that are not empty or '_'.
    Returns a list of newpart values for each row.
    """
    newparts = []
    last_newpart = "_"

    for value in df.get('newpart', pd.Series("_")).astype(str):
        value = value.strip()
        if value and value != "_":  # Only consider "real" markers
            last_newpart = value
        newparts.append(last_newpart)

    return newparts

def check_sense_issue(lemma, meaning, pos, lemma_dict, rec, new_lemma_added, new_sense_added):
    """
    Returns a list of (issue_type, description) tuples for the given lemma/meaning/pos.
    Updates:
        - new_lemma_added: set of new lemmata with context
        - new_sense_added: list of dicts with info about new senses
    """
    norm_lemma = _normalize_sense_text(lemma)
    norm_meaning_set = _normalize_verb_sense_text(meaning, pos)
    allowed_senses = lemma_dict.get(norm_lemma, set())
    reported = []
    stripped_lemma = _strip_trailing_digits(norm_lemma)

    # --- 1. Handle missing lemma cases ---
    if not allowed_senses:
        # Case A: Numbered variants like lemma1, lemma2
        numbered_variants = [
            lem for lem in lemma_dict.keys()
            if lem.startswith(norm_lemma) and re.search(r'\d+$', lem)
        ]

        matched_variant = None
        matched_sense = False

        if numbered_variants:
            # Try to match meaning to one of the variant’s senses
            for variant in numbered_variants:
                variant_senses = lemma_dict[variant]
                for norm_meaning in norm_meaning_set:
                    norm_meaning_stripped = norm_meaning.strip()
                    for allowed in variant_senses:
                        allowed_parts = [s.strip() for s in allowed.split('|')]
                        for part in allowed_parts:
                            if (
                                norm_meaning_stripped == part
                                or norm_meaning_stripped == part.replace('to ', '')
                                or (',' in part and norm_meaning_stripped in [x.strip() for x in part.split(',')])
                            ):
                                matched_variant = variant
                                matched_sense = True
                                break
                        if matched_sense:
                            break
                    if matched_sense:
                        break
                if matched_sense:
                    break

            if matched_variant:
                # Found matching sense in one variant → report it
                reported.append((
                    'GENERAL_IGNORING_EDITED_LEMMA',
                    f"Lemma '{lemma}' not found in dictionary, but matches variant '{matched_variant}'."
                ))
                allowed_senses = lemma_dict.get(matched_variant, set())

            else:
                # Variants exist but sense not found → record only in new_sense_added
                combined_senses = '|'.join(
                    sorted(set().union(*(lemma_dict[v] for v in numbered_variants)))
                )
                new_sense_added.append({
                    'file': rec.get('file'),
                    'row_index': rec.get('row_index'),
                    'lemma': lemma,
                    'upos': pos,
                    'new_sense': meaning,
                    'other_senses': combined_senses,
                    'newpart': rec.get('newpart', '_')  # Attach sentence metadata
                })
                allowed_senses = set().union(*(lemma_dict[v] for v in numbered_variants))

        elif stripped_lemma in lemma_dict and stripped_lemma != norm_lemma:
            # Case B: Base (non-numbered) lemma exists (e.g., run1 → run)
            reported.append((
                'GENERAL_IGNORING_EDITED_LEMMA',
                f"Lemma '{lemma}' not found in dictionary, but base form '{stripped_lemma}' exists."
            ))
            allowed_senses = lemma_dict.get(stripped_lemma, set())

        else:
            # Case C: Truly new lemma → goes to new_lemma_added
            new_lemma_added.add(
                f"{rec.get('file')} | Row {rec.get('row_index')} | Lemma: {lemma} | "
                f"UPOS: {pos} | Meaning: {meaning} | NEWPART: {rec.get('newpart', '_')}"
            )

    # --- 2. Meaning-level checks ---
    for norm_meaning in norm_meaning_set:
        if _is_placeholder_meaning(norm_meaning):
            continue

        norm_meaning_stripped = norm_meaning.strip()

        # 1. Exact pipe-separated match
        exact_match = any(
            norm_meaning_stripped in [s.strip() for s in allowed.split('|')]
            for allowed in allowed_senses
        )
        if exact_match:
            continue

        # 2. Verb with 'to' match
        if _is_verb_pos(pos):
            for allowed in allowed_senses:
                allowed_parts = [s.strip() for s in allowed.split('|')]
                for part in allowed_parts:
                    if norm_meaning_stripped == part.replace('to ', '') or norm_meaning_stripped == part:
                        reported.append(('VERB_INF_ISSUE',
                                         f"Verb sense with 'to': '{norm_meaning_stripped}' matches '{part}'"))
                        break
            if any(r[0] == 'VERB_INF_ISSUE' for r in reported):
                continue

        # 3. Comma-separated ignoring edited senses
        matched_ignoring_edited = False
        for allowed in allowed_senses:
            pipe_parts = [s.strip() for s in allowed.split('|')]
            for pipe in pipe_parts:
                if ',' in pipe:
                    comma_parts = [s.strip() for s in pipe.split(',')]
                    if norm_meaning_stripped in comma_parts:
                        matched_ignoring_edited = True
                        full_pipe_unit = pipe
                        if pos.upper() == 'ADP':
                            reported.append(('ADP_SENSE_IGNORING_EDITED',
                                             f"Senses between commas in pipe: '{norm_meaning_stripped}' matches full unit: '{full_pipe_unit}'"))
                        elif pos.upper() == 'ADV':
                            reported.append(('ADV_SENSE_IGNORING_EDITED',
                                             f"Senses between commas in pipe: '{norm_meaning_stripped}' matches full unit: '{full_pipe_unit}'"))
                        else:
                            reported.append(('GENERAL_IGNORING_EDITED_SENSE',
                                             f"Senses between commas in pipe: '{norm_meaning_stripped}' matches full unit: '{full_pipe_unit}'"))
                        break
            if matched_ignoring_edited:
                break
        if matched_ignoring_edited:
            continue

        # 4. Verb 'to' + comma-separated ignoring edited
        if _is_verb_pos(pos):
            for allowed in allowed_senses:
                pipe_parts = [s.strip() for s in allowed.split('|')]
                for pipe in pipe_parts:
                    comma_parts = [s.strip() for s in pipe.split(',')]
                    for part in comma_parts:
                        if f"to {norm_meaning_stripped}" == part or norm_meaning_stripped == part.replace('to ', ''):
                            full_pipe_unit = pipe
                            reported.append(('VERB_INF_IGNORING_EDITED',
                                             f"Verb ignoring edited sense: '{norm_meaning_stripped}' matches full unit: '{full_pipe_unit}'"))
                            matched_ignoring_edited = True
                            break
                    if any(r[0] == 'VERB_INF_IGNORING_EDITED' for r in reported):
                        break
                if any(r[0] == 'VERB_INF_IGNORING_EDITED' for r in reported):
                    break

        # 5. If nothing matched — lemma-sense mismatch
        if not any(r[0] != 'LEMMA_SENSE_MISMATCH' for r in reported):
            reported.append(('LEMMA_SENSE_MISMATCH',
                             f"Meaning '{norm_meaning_stripped}' not found among allowed senses for lemma '{lemma}'"))
            # Record as new sense if lemma already exists
            if norm_lemma in lemma_dict:
                full_cell_matches = any(
                    norm_meaning_stripped == _normalize_sense_text(cell).strip() or
                    (_is_verb_pos(pos) and f"to {norm_meaning_stripped}" == _normalize_sense_text(cell).strip())
                    for cell in lemma_dict.get(norm_lemma, [])
                )
                if not full_cell_matches:
                    new_sense_added.append({
                        'file': rec.get('file'),
                        'row_index': rec.get('row_index'),
                        'lemma': lemma,
                        'upos': pos,
                        'new_sense': norm_meaning_stripped,
                        'other_senses': '|'.join(lemma_dict.get(norm_lemma, [])),
                        'newpart': rec.get('newpart', '_')  # Attach sentence metadata
                    })

    return reported

def load_lemma_dictionary(dict_path):
    """
    Load lemma → allowed senses from a CSV/Excel dictionary.

    Handles:
    - Multi-sense entries separated by '|' or ','
    - Numeric suffixes in lemmata
    """
    import pandas as pd
    df = pd.read_excel(dict_path, dtype=str, keep_default_na=False, engine='openpyxl') \
        if dict_path.lower().endswith(('.xls', '.xlsx')) else pd.read_csv(dict_path, dtype=str, keep_default_na=False)
    
    lemma_col = next((c for c in df.columns if 'lemma' in c.lower()), None)
    sense_col = next((c for c in df.columns if 'sense' in c.lower() or 'meaning' in c.lower()), None)
    if not lemma_col or not sense_col:
        raise ValueError("Dictionary must have lemma and sense columns")
    
    lemma_map = {}
    for _, row in df.iterrows():
        lemma = _normalize_sense_text(row.get(lemma_col, ''))
        senses_raw = row.get(sense_col, '')
        if not senses_raw or str(senses_raw).strip() == '':
            senses = set()
        else:
            parts = re.split(r'\s*\|\s*|\s*,\s*', str(senses_raw))
            senses = {_normalize_sense_text(p) for p in parts if p.strip()}
        if lemma:
            lemma_map[lemma] = senses
    return lemma_map
def _normalize_sense_text(text):
    """
    Normalize a lemma or sense string for consistent comparison.

    This function:
      - Converts the input to string (if not already)
      - Applies Unicode normalization (NFC)
      - Strips leading/trailing whitespace
      - Converts to lowercase

    Used for:
      - Normalizing lemma names when reading from dictionary files
      - Normalizing sense entries (e.g., meanings like "This" → "this")

    Parameters
    ----------
    text : str
        Input text (lemma or sense)

    Returns
    -------
    str
        A clean, normalized version of the input suitable for
        dictionary comparison and equality checks.
    """
    if text is None:
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()


def analyze_corpus(input_dir, lemma_dict_path=None, report_lemma_dir=None,
                   report_other_dir=None, consolidated_lemma_path=None):
    lemma_dict = load_lemma_dictionary(lemma_dict_path) if lemma_dict_path else {}
    # --- Load consolidated lemma list ---
    consolidated_lemmata = set()
    if consolidated_lemma_path and os.path.exists(consolidated_lemma_path):
        with open(consolidated_lemma_path, 'r', encoding='utf-8') as f:
            consolidated_lemmata = {line.strip() for line in f if line.strip()}
        print(f"Loaded {len(consolidated_lemmata)} consolidated lemmata.")

    files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(CSV_EXTS) and not f.startswith('~$'))
    all_issues = []
    new_lemma_added = set()
    new_sense_added = []
    consolidated_matches = []  # For consolidated lemma matches


    for fname in files:
        path = os.path.join(input_dir, fname)
        ext = os.path.splitext(path)[1].lower()
        try:
            df = pd.read_excel(path, dtype=str, keep_default_na=False, engine='openpyxl') if ext in ('.xlsx', '.xls') \
                else pd.read_csv(path, sep='\t' if path.endswith('.tsv') else ',', dtype=str, keep_default_na=False, on_bad_lines='skip')
        except Exception:
            continue

        # --- Initialize per-file issue lists BEFORE processing records ---
        lemma_issues = []
        verb_inf_issues = []
        verb_inf_ignoring_edited = []
        adp_sense_ignoring_edited = []
        adv_sense_ignoring_edited = []
        general_ignoring_edited = []
        general_ignoring_edited_lemma = []
        other_issues = []

        records = []
        def col_getter(row, *cols):
            for c in cols:
                if c in row and str(row[c]).strip() != '':
                    return str(row[c])
            return ''

        trans_cols = ('transcription', 'form', 'text', 'trans')
        lemma_cols = ('lemma', 'Lemma')
        translit_cols = ('transliteration', 'translit')
        pos_cols = ('postag', 'upos', 'pos')
        feats_cols = ('postfeatures', 'features', 'feats')
        trans_meaning_cols = ('translation', 'meaning')

        # --- Assign newpart values to all rows, once per file ---
        newparts = assign_newpart(df)

        for idx, row in df.iterrows():
            transcription = col_getter(row, *trans_cols)
            lemma = col_getter(row, *lemma_cols)
            translit = _normalize_translit_backslash(col_getter(row, *translit_cols))
            pos = col_getter(row, *pos_cols)
            feats = col_getter(row, *feats_cols)
            meaning = col_getter(row, *trans_meaning_cols)
            
            rec = {'file': fname, 'row_index': idx, 'transcription': transcription, 'lemma': lemma,
                'transliteration': translit, 'pos': pos, 'features': feats, 'meaning': meaning,
                'newpart': newparts[idx]}

            if _is_exception_token(transcription) or _is_exception_token(translit) or _is_exception_token(lemma):
                continue

            # --- Trailing-space check ---
            for col_name in ['transcription', 'lemma', 'transliteration', 'meaning']:
                value = rec.get(col_name, '')
                if value and value != value.rstrip():
                    other_issues.append(dict(rec,
                                            issue_type='TRAILING_SPACE',
                                            issue_desc=f"Trailing space found in column '{col_name}': '{value}'"))

            records.append(rec)

        # --- Initialize per-file issue lists ---
        lemma_issues = []
        verb_inf_issues = []
        verb_inf_ignoring_edited = []
        adp_sense_ignoring_edited = []
        adv_sense_ignoring_edited = []
        general_ignoring_edited = []
        general_ignoring_edited_lemma = []
        other_issues = []

        # --- Process records ---
        for rec in records:
            transcription = rec['transcription']
            lemma = rec['lemma']
            translit = rec['transliteration']
            pos = rec['pos']
            feats = rec['features'] or ''
            meaning = rec['meaning']

            if lemma and meaning and lemma_dict:
                results = check_sense_issue(lemma, meaning, pos, lemma_dict, rec, new_lemma_added, new_sense_added)
                for issue_type, issue_desc in results:
                    if issue_type == 'LEMMA_SENSE_MISMATCH':
                        lemma_issues.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'VERB_INF_ISSUE':
                        verb_inf_issues.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'VERB_INF_IGNORING_EDITED':
                        verb_inf_ignoring_edited.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'ADP_SENSE_IGNORING_EDITED':
                        adp_sense_ignoring_edited.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'ADV_SENSE_IGNORING_EDITED':
                        adv_sense_ignoring_edited.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'GENERAL_IGNORING_EDITED':
                        general_ignoring_edited.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
                    elif issue_type == 'GENERAL_IGNORING_EDITED_LEMMA':
                        general_ignoring_edited_lemma.append(dict(rec, issue_type=issue_type, issue_desc=issue_desc))
            
            # --- CONSOLIDATED LEMMA MATCH CHECK ---
            if lemma and lemma.strip() in consolidated_lemmata:
                consolidated_matches.append({
                    'file': rec['file'],
                    'row_index': rec['row_index'],
                    'lemma': lemma,
                    'meaning': meaning,
                    'newpart': rec['newpart'],
                    'description': f"Match for lemma '{lemma}' with sense '{meaning}' found in NEWPART '{rec['newpart']}'."
                })


            # Other annotation checks...
            if _has_disallowed_punct(transcription) and not transcription.startswith('#'):
                other_issues.append(dict(rec, issue_type='PUNCT_IN_TRANSCRIPTION', issue_desc='Disallowed punctuation.'))
            if _ends_with_comparative(transcription) and lemma and re.search(r'(dar|tar)$', lemma):
                other_issues.append(dict(rec, issue_type='COMPARATIVE_IN_LEMMA', issue_desc='Comparative suffix in lemma.'))
            if _is_verb_pos(pos) and _lemma_missing_infinitive(lemma, transcription, pos):
                other_issues.append(dict(rec, issue_type='MISSING_INFINITIVE_IN_LEMMA', issue_desc='Missing infinitive in lemma.'))
            if _enclitic_hyphen_issue(transcription):
                other_issues.append(dict(rec, issue_type='ENCLITIC_HYPHEN_ISSUE', issue_desc='Enclitic hyphen problem.'))
            if '_' in transcription and (lemma == '' or '_' in lemma):
                other_issues.append(dict(rec, issue_type='COMPOUND_UNDERSCORE_ISSUE', issue_desc='Compound underscore issue.'))
           
            if _wordend_marker_inconsistent(translit):
                other_issues.append(dict(rec, issue_type='TRANSLIT_WORDEND_INCONSISTENT', issue_desc='Inconsistent word-end markers.'))
            if _has_restored_or_question(translit):
                other_issues.append(dict(rec, issue_type='RESTORED_OR_QUESTION_IN_TRANSLIT', issue_desc='Restored/uncertain characters (?, *, []).'))

        #
        def _write_report(issues_list, folder, prefix, fname):
            """
            Writes annotation issues in a linear, readable format for each row.
            Each entry includes NEWPART.
            """
            if not folder:
                return
            os.makedirs(folder, exist_ok=True)

            filename = f"{prefix}_{os.path.splitext(fname)[0]}.txt" if issues_list else f"{prefix}_{os.path.splitext(fname)[0]}_empty.txt"
            path = os.path.join(folder, filename)

            with open(path, 'w', encoding='utf-8') as f:
                if not issues_list:
                    f.write("✅ No issues detected.\n")
                    return

                f.write(f"Issues Report: {prefix} ({fname})\n")
                f.write("="*80 + "\n\n")

                for i, issue in enumerate(issues_list, 1):
                    f.write(f"Entry {i}\n")
                    f.write(f"Row: {issue.get('row_index','')}\n")
                    f.write(f"File: {issue.get('file','')}\n")
                    f.write(f"Lemma: {issue.get('lemma','')}\n")
                    f.write(f"POS: {issue.get('pos','')}\n")
                    f.write(f"Transcription: {issue.get('transcription','')}\n")
                    f.write(f"Transliteration: {issue.get('transliteration','')}\n")
                    f.write(f"Features: {issue.get('features','')}\n")
                    f.write(f"Meaning: {issue.get('meaning','')}\n")
                    f.write(f"NEWPART: {issue.get('newpart','')}\n")  # Include NEWPART
                    f.write(f"Issue Type: {issue.get('issue_type','')}\n")
                    f.write(f"Description: {issue.get('issue_desc','')}\n")
                    f.write("-"*60 + "\n\n")


        def _write_set_report(s, folder, prefix):
            """
            Writes new lemma entries in a clear, linear format.
            Each entry includes NEWPART.
            """
            if not folder:
                return
            os.makedirs(folder, exist_ok=True)

            filename = f"{prefix}.txt" if s else f"{prefix}_empty.txt"
            path = os.path.join(folder, filename)

            with open(path, 'w', encoding='utf-8') as f:
                if not s:
                    f.write("✅ No new lemmata detected.\n")
                    return

                f.write(f"New Lemmata Report: {prefix}\n")
                f.write("="*80 + "\n\n")

                for i, item in enumerate(sorted(s), 1):
                    parts = item.split('|')
                    f.write(f"Entry {i}\n")
                    f.write(f"File: {parts[0].strip()}\n")
                    f.write(f"Row: {parts[1].replace('Row','').strip()}\n")
                    f.write(f"Lemma: {parts[2].replace('Lemma:','').strip()}\n")
                    f.write(f"POS: {parts[3].replace('UPOS:','').strip()}\n")
                    f.write(f"Meaning: {parts[4].replace('Meaning:','').strip()}\n")
                    f.write(f"NEWPART: {parts[5].replace('NEWPART:','').strip() if len(parts) > 5 else '_'}\n")  # Include NEWPART
                    f.write("-"*60 + "\n\n")


        def _write_new_sense_report(s, folder, prefix):
            """
            Writes new senses in a linear, readable format.
            Each entry includes NEWPART.
            """
            if not folder:
                return
            os.makedirs(folder, exist_ok=True)

            filename = f"{prefix}.txt" if s else f"{prefix}_empty.txt"
            path = os.path.join(folder, filename)

            with open(path, 'w', encoding='utf-8') as f:
                if not s:
                    f.write("✅ No new senses detected.\n")
                    return

                f.write(f"New Senses Report: {prefix}\n")
                f.write("="*80 + "\n\n")

                for i, issue in enumerate(s, 1):
                    f.write(f"Entry {i}\n")
                    f.write(f"Row: {issue.get('row_index','')}\n")
                    f.write(f"File: {issue.get('file','')}\n")
                    f.write(f"Lemma: {issue.get('lemma','')}\n")
                    f.write(f"POS: {issue.get('upos','')}\n")
                    f.write(f"New Sense: {issue.get('new_sense','')}\n")
                    f.write(f"Other Senses: {issue.get('other_senses','')}\n")
                    f.write(f"NEWPART: {issue.get('newpart','')}\n")  # Include NEWPART
                    f.write("-"*60 + "\n\n")


        # Write reports
        #_write_report(lemma_issues, report_lemma_dir, "dictionary_issues", fname)
        _write_report(verb_inf_issues, report_lemma_dir, "verb_inf_issues", fname)
        _write_report(verb_inf_ignoring_edited, report_lemma_dir, "verb_inf_ignoring_edited", fname)
        _write_report(adp_sense_ignoring_edited, report_lemma_dir, "adp_sense_ignoring_edited", fname)
        _write_report(adv_sense_ignoring_edited, report_lemma_dir, "adv_sense_ignoring_edited", fname)
        _write_report(general_ignoring_edited, report_lemma_dir, "general_ignoring_edited", fname)
        _write_report(general_ignoring_edited_lemma, report_lemma_dir, "general_ignoring_edited_lemma", fname)
        _write_report(other_issues, report_other_dir, "other_issues", fname)

        # For new lemma and new sense reports, do NOT pass fname
        _write_set_report(new_lemma_added, report_lemma_dir, "new_lemmata")
        _write_new_sense_report(new_sense_added, report_lemma_dir, "new_senses")


        all_issues.extend(lemma_issues + verb_inf_issues + verb_inf_ignoring_edited +
                          adp_sense_ignoring_edited + adv_sense_ignoring_edited + general_ignoring_edited +
                          other_issues)

    def consolidated_lemmata (matches, folder, prefix="consolidated_matches"):
        """
        Writes the consolidated lemma matches grouped by NEWPART.
        """
        if not folder:
            return
        os.makedirs(folder, exist_ok=True)
        filename = f"{prefix}.txt" if matches else f"{prefix}_empty.txt"
        path = os.path.join(folder, filename)

        with open(path, 'w', encoding='utf-8') as f:
            if not matches:
                f.write("✅ No matches found in consolidated lemma list.\n")
                return

            f.write(f"Consolidated Lemma Match Report (Grouped by NEWPART)\n{'='*80}\n\n")

            # --- Group matches by NEWPART ---
            from collections import defaultdict
            grouped = defaultdict(list)
            for m in matches:
                grouped[m.get('newpart', '_')].append(m)

            # --- Write each NEWPART section ---
            for newpart, group in sorted(grouped.items(), key=lambda x: str(x[0])):
                f.write(f"### NEWPART: {newpart}\n")
                f.write("-"*80 + "\n")
                for i, m in enumerate(group, 1):
                    f.write(f"Entry {i}\n")
                    f.write(f"File: {m['file']}\n")
                    f.write(f"Row: {m['row_index']}\n")
                    f.write(f"Lemma: {m['lemma']}\n")
                    f.write(f"Meaning: {m['meaning']}\n")
                    f.write(f"Description: {m['description']}\n")
                    f.write("-"*60 + "\n")
                f.write("\n")

        print(f"✅ Consolidated lemma report written (grouped by NEWPART) to: {path}")


    return pd.DataFrame(all_issues)


In [2]:
# First ensure load_lemma_dictionary is correctly defined

df_report = analyze_corpus(
    input_dir=r"C:\Users\rahaa\Dropbox\MPCD\test_for_input",
    lemma_dict_path=r"C:\Users\rahaa\Dropbox\MPCD\dict_export_17-10-2025.xlsx",
    report_lemma_dir=r"C:\Users\rahaa\Dropbox\MPCD\test_for_input\lemma_reports",
    report_other_dir=r"C:\Users\rahaa\Dropbox\MPCD\test_for_input\other_reports",
    consolidated_lemma_path=r"C:\Users\rahaa\Dropbox\MPCD\consolidated_lemmata.txt"

)


Loaded 841 consolidated lemmata.
